# Vessel-Enhanced Deep Learning for ADAM Dataset Aneurysm Detection

This notebook contains the pipeline to load the **ADAM dataset** (MICCAI Aneurysm Detection And segMentation Challenge) from Google Drive, split it patient-wise into **70% Train, 15% Validation, and 15% Test** sets, and train/evaluate the model using PyTorch.

### Pipeline Overview:
1. **Google Drive Mount**: Connects Colab to your Google Drive files.
2. **Dataset Unzipping**: Automatically extracts the ADAM dataset onto Colab's fast local SSD.
3. **ADAM Path Matching**: Traverses patient folders (`10001`, `10002`...) to pair `{patient_id}/pre/TOF.nii.gz` scans with `{patient_id}/aneurysms.nii.gz` masks.
4. **Dataset Sanity Check**: Verifies NIfTI shapes and voxel alignments.
5. **3D Patched Loader**: Loads NIfTI volumes and extracts balanced 3D patches.
6. **Preprocessed Patch Sanity Check**: Verifies that 3D patches are correct.
7. **Vessel-Enhanced Training**: Trains the networks end-to-end.
8. **Final Evaluation**: Evaluates on the unseen 15% test set.
9. **Hyperparameter Tuning (Optuna)**: Optional optimization section.

## Step 1: Connect to Google Drive & Setup GPU
Run this cell to mount your Google Drive and verify GPU availability.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Go to 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU' to run training on GPU.")

In [ ]:
# Install nibabel to read NIfTI file formats (.nii or .nii.gz) and optuna for tuning
!pip install nibabel scipy matplotlib optuna

## Step 2: Unzip Dataset to Colab SSD
Update the path below to point to the location of your uploaded ADAM dataset ZIP file on Google Drive, then run the cell.

In [ ]:
import os
import zipfile
import glob

# Path to your main ZIP file on Google Drive (e.g. 'ADAM.zip')
ZIP_PATH = '/content/drive/MyDrive/path_to_your_ADAM_dataset.zip'

# Destination on Colab's fast local SSD
LOCAL_DESTINATION = '/content/dataset'

def extract_zip_recursive(zip_path, extract_to):
    print(f"Extracting {os.path.basename(zip_path)} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    
    # Search for nested zip files inside the extracted folder
    nested_zips = glob.glob(os.path.join(extract_to, "**/*.zip"), recursive=True)
    for nested_zip in nested_zips:
        nested_dest = os.path.dirname(nested_zip)
        print(f"  -> Found nested zip: {os.path.basename(nested_zip)}. Extracting...")
        with zipfile.ZipFile(nested_zip, 'r') as nested_ref:
            nested_ref.extractall(nested_dest)
        os.remove(nested_zip)

if os.path.exists(ZIP_PATH):
    os.makedirs(LOCAL_DESTINATION, exist_ok=True)
    extract_zip_recursive(ZIP_PATH, LOCAL_DESTINATION)
    print("\nAll extractions complete! Your files are ready.")
else:
    print(f"ERROR: ZIP file not found at path: {ZIP_PATH}")

## Step 2.5: Configure Dataset Root Path
Specify the folder path containing your unzipped patient folders (`10001`, `10002`...).

In [ ]:
# This is the root folder containing the patient folders (10001, 10002...)
# Adjust if the folder structure inside your zip has an extra nested directory (e.g. '/content/dataset/ADAM_ready')
DATASET_ROOT = '/content/dataset/ADAM_ready'

print(f"Dataset root path set to: {DATASET_ROOT}")
print(f"Path exists: {os.path.exists(DATASET_ROOT)}")
if os.path.exists(DATASET_ROOT):
    print("Sample of files in root:", os.listdir(DATASET_ROOT)[:5])

## Step 3: Parse ADAM Folder Structure & Split Patients (70/15/15)
Instead of flat folders, we traverse each patient folder to locate:
- **Image:** `{patient_id}/pre/TOF.nii.gz`
- **Mask:** `{patient_id}/aneurysms.nii.gz`

In [ ]:
import glob
import random
from dataset import AneurysmDataset
from torch.utils.data import DataLoader

# 1. Find all numerical patient folders (e.g., 10001, 10002...)
patient_folders = sorted([
    f for f in glob.glob(os.path.join(DATASET_ROOT, "*")) 
    if os.path.isdir(f) and os.path.basename(f).isdigit()
])

# 2. Scan each patient folder and pair TOF scan with aneurysm mask
file_list = []
for folder in patient_folders:
    patient_id = os.path.basename(folder)
    
    image_path = os.path.join(folder, 'pre', 'TOF.nii.gz')
    mask_path = os.path.join(folder, 'aneurysms.nii.gz')
    
    if os.path.exists(image_path) and os.path.exists(mask_path):
        file_list.append({
            'image': image_path,
            'label': mask_path,
            'patient_id': patient_id
        })
    else:
        print(f"Warning: Missing files for Patient {patient_id}.")
        print(f"  -> TOF exists: {os.path.exists(image_path)} | Mask exists: {os.path.exists(mask_path)}")

print(f"\nSuccessfully paired {len(file_list)} ADAM patient scans.")
if len(file_list) == 0:
    raise ValueError("No matched image-mask pairs found. Please verify the folder structure in DATASET_ROOT.")

# 3. Split dataset using specific test set patient IDs
from dataset import split_dataset_by_test_patients

test_patient_ids = [
    '10027', '10030', '10031', '10032', '10043',
    '10044B', '10048B', '10053B', '10053F', '10054F',
    '10058F', '10059B', '10059F', '10070F', '10066F',
    '10073B', '10073F'
]

train_files, val_files, test_files = split_dataset_by_test_patients(file_list, test_patient_ids)

print(f"\nPatient Split Results:")
print(f"  Training set:   {len(train_files)} patients")
print(f"  Validation set: {len(val_files)} patients")
print(f"  Test set:       {len(test_files)} patients")

## Step 4: Clinical Data Sanity Check
Before training, run this cell to perform a sanity check on a patient scan. It verifies that:
1. The raw MRA scan and mask dimensions match exactly.
2. The mask contains aneurysm voxels (is not empty).
3. Voxel spatial grids align correctly, and plots a 2D cross-section overlay showing the aneurysm location on the scan.

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

# Grab the first patient pair for checking
check_pair = file_list[0]
print("Checking Patient Folder:", check_pair['patient_id'])
print("Checking Scan:", os.path.basename(check_pair['image']))
print("Checking Mask:", os.path.basename(check_pair['label']))

# 1. Load volumes
img_obj = nib.load(check_pair['image'])
lbl_obj = nib.load(check_pair['label'])

img_data = img_obj.get_fdata()
lbl_data = lbl_obj.get_fdata()

# 2. Dimension checks
print(f"\n1. Dimension Verification:")
print(f"  Raw Scan Shape: {img_data.shape}")
print(f"  Mask Shape:     {lbl_data.shape}")
assert img_data.shape == lbl_data.shape, "CRITICAL ERROR: Scan and mask sizes do not match!"
print("  => SUCCESS: Shapes match.")

# 3. Affine/grid alignment checks
print(f"\n2. Spatial Orientation Verification:")
img_affine = img_obj.affine
lbl_affine = lbl_obj.affine
shapes_match = np.allclose(img_affine, lbl_affine, atol=1e-3)
if shapes_match:
    print("  => SUCCESS: Affine matrices match. Grid alignment is correct.")
else:
    print("  => WARNING: Affine matrices differ slightly. Make sure orientation and voxel grids are aligned!")

# 4. Aneurysm Presence
num_aneurysm_voxels = np.sum(lbl_data > 0.5)
print(f"\n3. Label Check:")
print(f"  Aneurysm Voxels Labeled: {int(num_aneurysm_voxels)}")
assert num_aneurysm_voxels > 0, "ERROR: Mask contains no aneurysm (all zeros). Check segmentation file!"

# 5. Visualize visual alignment
# Find coordinate of the aneurysm center to plot a slice through it
an_voxels = np.argwhere(lbl_data > 0.5)
center_z, center_y, center_x = an_voxels[len(an_voxels) // 2]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot raw slice
axes[0].imshow(img_data[center_z], cmap='gray')
axes[0].set_title(f"Raw Scan Slice {center_z}")
axes[0].axis('off')

# Plot mask slice
axes[1].imshow(lbl_data[center_z], cmap='gray')
axes[1].set_title("Mask Slice")
axes[1].axis('off')

# Overlay plot
axes[2].imshow(img_data[center_z], cmap='gray')
axes[2].imshow(lbl_data[center_z], cmap='red', alpha=0.5) # Red overlay
axes[2].set_title("Vessel/Aneurysm Overlay (Check alignment)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Step 5: Initialize Datasets & Dataloaders
Initialize datasets and dataloaders using the verified splits.

In [ ]:
# 4. Initialize Datasets
# Note on A100 GPU: Since A100 has large VRAM (40GB/80GB), you can increase patch_size 
# (e.g. to 96x96x96) to give the model more context, and increase num_patches_per_volume.
train_dataset = AneurysmDataset(
    file_list=train_files, 
    patch_size=(64, 64, 64), 
    num_patches_per_volume=4
)
val_dataset = AneurysmDataset(
    file_list=val_files, 
    patch_size=(64, 64, 64), 
    num_patches_per_volume=2
)
test_dataset = AneurysmDataset(
    file_list=test_files, 
    patch_size=(64, 64, 64), 
    num_patches_per_volume=2
)

# 5. Dataloaders
# Note on A100: You can increase batch_size to 16, 32, or 64. 
# We enable pin_memory=True to speed up CPU-to-GPU data transfers.
BATCH_SIZE = 16  # Scaled up for A100 VRAM (default was 4)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## Step 6: Preprocessed Patch Sanity Check
Before training, run this cell to verify that the **preprocessed 3D patches** (the actual tensors fed into the neural network) are correct. This checks:
1. **Tensor Shape:** Checks batch, channel, and spatial dimensions.
2. **NaN / Inf Check:** Ensures no bad pixel values exist (which would crash training).
3. **Normalization Bounds:** Displays range statistics (min, max, mean, std).
4. **Target Inclusion:** Verifies that training patches successfully contain the aneurysm mask at the center.

In [ ]:
# Fetch a single batch from the training loader
train_iter = iter(train_loader)
images_batch, targets_batch = next(train_iter)

print("--- Preprocessed Batch Statistics ---")
print(f"Raw Scan Batch Shape: {images_batch.shape}  (Expected: [Batch_Size, 1, Depth, Height, Width])")
print(f"Mask Batch Shape:     {targets_batch.shape}  (Expected: [Batch_Size, 1, Depth, Height, Width])")

# 1. NaN and Inf Check
has_nan_img = torch.isnan(images_batch).any().item()
has_inf_img = torch.isinf(images_batch).any().item()
has_nan_lbl = torch.isnan(targets_batch).any().item()
print(f"\n1. Value Integrity Check:")
print(f"  Images contain NaNs: {has_nan_img} | Infs: {has_inf_img}")
print(f"  Masks contain NaNs:  {has_nan_lbl}")
assert not (has_nan_img or has_inf_img or has_nan_lbl), "CRITICAL ERROR: Found NaNs or Infs in data batch!"
print("  => SUCCESS: Value integrity check passed.")

# 2. Voxel Normalization Check
img_min = images_batch.min().item()
img_max = images_batch.max().item()
img_mean = images_batch.mean().item()
img_std = images_batch.std().item()
print(f"\n2. Normalization Bounds (Z-Score):")
print(f"  Min: {img_min:.4f} | Max: {img_max:.4f}")
print(f"  Mean: {img_mean:.4f} | Std: {img_std:.4f}")

# 3. Target Distribution Check
mask_max = targets_batch.max().item()
mask_min = targets_batch.min().item()
assert mask_max == 1.0 and mask_min == 0.0, f"ERROR: Target mask should be binary (0 or 1), got min={mask_min}, max={mask_max}"
print(f"\n3. Target Distribution:")
print("  => SUCCESS: Mask is binary [0, 1].")

# 4. Visual Verification (Slice 32 of Patch 0)
patch_idx = 0
slice_z = images_batch.shape[2] // 2
img_slice = images_batch[patch_idx, 0, slice_z].numpy()
lbl_slice = targets_batch[patch_idx, 0, slice_z].numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_slice, cmap='gray')
axes[0].set_title(f"Preprocessed Patch Slice {slice_z}")
axes[0].axis('off')

axes[1].imshow(lbl_slice, cmap='gray')
axes[1].set_title("Preprocessed Mask Slice")
axes[1].axis('off')

axes[2].imshow(img_slice, cmap='gray')
axes[2].imshow(lbl_slice, cmap='red', alpha=0.4)
axes[2].set_title("Overlay (Verify aneurysm is centered)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Step 7: Define Model & Run Training
We initialize our model (`VesselEnhancedAneurysmNet`) and begin training using the `AneurysmTrainer` wrapper. Training history will be saved.

In [ ]:
from model import VesselEnhancedAneurysmNet
from train import AneurysmTrainer

# Initialize model
model = VesselEnhancedAneurysmNet()

# Optional: Optimize PyTorch compilation for A100 Ampere GPU architecture (PyTorch 2.0+)
# Uncomment the line below to speed up the training loop by ~20%
# model = torch.compile(model)

# Initialize trainer
# use_amp=True enables Automatic Mixed Precision (FP16) which optimizes A100 compute speed
trainer = AneurysmTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=1e-3,
    device='cuda', 
    checkpoint_path='/content/drive/MyDrive/best_aneurysm_model.pth', 
    use_amp=True 
)

# Train for 30 epochs (Increase this for better convergence, e.g., 50-100)
print("Starting training loop...")
history = trainer.fit(epochs=30)

## Step 8: Plot Training Loss & Metrics

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
plt.plot(epochs, history['val_loss'], 'r-s', label='Val Loss')
plt.title('Loss Curves (Dice-BCE)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Dice Score
plt.subplot(1, 2, 2)
plt.plot(epochs, history['train_dice'], 'b-o', label='Train Dice')
plt.plot(epochs, history['val_dice'], 'r-s', label='Val Dice')
plt.title('Aneurysm Segment Dice Coefficient')
plt.xlabel('Epochs')
plt.ylabel('Dice Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Step 9: Final Evaluation on Unseen Test Dataset
Now we load the best saved model weights and run an evaluation on the completely unseen **Test Set** (15% of the data).

In [ ]:
from train import compute_metrics

# 1. Load the saved weights
checkpoint = torch.load('/content/drive/MyDrive/best_aneurysm_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.to(trainer.device)
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']} with Val Dice {checkpoint['val_dice']:.4f}")

# 2. Run test loop
test_metrics_sum = {'dice': 0.0, 'sensitivity': 0.0, 'specificity': 0.0}

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(trainer.device)
        targets = targets.to(trainer.device)
        
        with torch.cuda.amp.autocast(enabled=trainer.use_amp):
            logits, vesselness = model(images)
        
        batch_metrics = compute_metrics(logits, targets)
        for k in test_metrics_sum:
            test_metrics_sum[k] += batch_metrics[k]

num_test_batches = len(test_loader)
final_test_metrics = {k: v / num_test_batches for k, v in test_metrics_sum.items()}

print("\n--- Final Test Set Results ---")
print(f"  Test Dice Score:  {final_test_metrics['dice']:.4f}")
print(f"  Test Sensitivity: {final_test_metrics['sensitivity']:.4f} (Ability to find aneurysms)")
print(f"  Test Specificity: {final_test_metrics['specificity']:.4f} (Ability to avoid false positives)")

## Step 10: Visualizing Test Sample Predictions
Let's visualize the results on a test sample. We take a 2D slice right through the center (slice 32) of a 3D patch and compare the raw MRA scan, the intermediate vessel-enhanced map, the ground truth aneurysm mask, and the model's predictions.

In [ ]:
import numpy as np

# 1. Grab a single test sample
image, label = test_dataset[0] # Shapes: [1, 64, 64, 64]

# 2. Pass it through the model
with torch.no_grad():
    input_tensor = image.unsqueeze(0).to(trainer.device)
    with torch.cuda.amp.autocast(enabled=trainer.use_amp):
        logits, vesselness = model(input_tensor)
    probs = torch.sigmoid(logits)
    
    # Convert outputs to NumPy arrays
    image_np = image.squeeze().cpu().numpy()
    label_np = label.squeeze().cpu().numpy()
    vesselness_np = vesselness.squeeze().cpu().numpy()
    probs_np = probs.squeeze().cpu().numpy()

# 3. Select the middle slice along the depth axis
slice_idx = 32

# 4. Plot results
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Raw MRA image
axes[0].imshow(image_np[slice_idx], cmap='gray')
axes[0].set_title(f"Raw MRA (Slice {slice_idx})")
axes[0].axis('off')

# Vesselness map generated by Vesselness Subnet
axes[1].imshow(vesselness_np[slice_idx], cmap='hot')
axes[1].set_title("Vessel Enhancement Map")
axes[1].axis('off')

# Ground truth aneurysm mask
axes[2].imshow(label_np[slice_idx], cmap='gray')
axes[2].set_title("Ground Truth Aneurysm")
axes[2].axis('off')

# Model prediction probabilities
axes[3].imshow(probs_np[slice_idx], cmap='jet', vmin=0, vmax=1)
axes[3].set_title("Aneurysm Probability Map")
axes[3].axis('off')

plt.tight_layout()
plt.show()

## Step 11: Automated Hyperparameter Tuning (Optional)
If you want to maximize model performance, you can use **Optuna** (a Bayesian hyperparameter optimization framework) to automatically search for the best learning rate and loss weights.

In [ ]:
import optuna
from train import DiceBCELoss

def objective(trial):
    # 1. Suggest hyperparameter values to test
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    bce_weight = trial.suggest_float("bce_weight", 0.1, 0.9)
    
    # 2. Re-initialize model and trainer with suggested values
    model_tune = VesselEnhancedAneurysmNet()
    
    trainer_tune = AneurysmTrainer(
        model=model_tune,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=lr,
        device='cuda',
        checkpoint_path='/content/drive/MyDrive/temp_tune.pth',
        use_amp=True
        )
    # Set the custom BCE loss weight
    trainer_tune.criterion = DiceBCELoss(bce_weight=bce_weight)
    
    # 3. Train for a few epochs (e.g., 5 epochs per trial for speed)
    print(f"\n--- Starting Trial {trial.number} --- (LR: {lr:.5f}, BCE Weight: {bce_weight:.2f})")
    history = trainer_tune.fit(epochs=5)
    
    # 4. Return best validation Dice score found in this trial
    best_dice = max(history['val_dice'])
    return best_dice

# Create and run the study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5) # Adjust n_trials as needed (e.g., 10-20)

print("\n--- Hyperparameter Optimization Complete ---")
print("Best Trial Results:")
print(f"  Val Dice Score: {study.best_trial.value:.4f}")
print("Best Hyperparameters:")
for key, val in study.best_trial.params.items():
    print(f"  {key}: {val}")